# Feedback Outcome Decoding

This notebook reads saved feedback-locked epochs with metadata and compares outcome decodability across task contexts, using `decoding.epoch_io.py`

In [2]:
from pathlib import Path
import sys

if repo_root.name == 'scripts':
    repo_root = repo_root.parent
if not (repo_root / 'scripts').exists():
    raise RuntimeError(f'Could not locate repo root from {repo_root}')

sys.path.insert(0, str(repo_root))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import mne
mne.set_log_level('WARNING')

from scripts import config
from scripts.decoding.epoch_io import get_epochs_path, load_epochs
from scripts.decoding.window_decoding_utils import (
    run_group_decoding_window, 
    save_outputs_window,
)
output_dir = repo_root / 'output_mne' / 'decoding'

NameError: name 'repo_root' is not defined

In [ ]:
PIPELINE = 'proposed'
SUBJECTS = sorted(config.SUBJECT_INFO.keys())
CONTEXTS = ['mid_high', 'high_high']
WINDOW_START = 0.20
WINDOW_END = 0.35

# dchecking the first 5 subjects
SUBJECTS[:5]

In [ ]:
# sanity check: load one subject's epochs and look at the metadata
example_subject = SUBJECTS[0]
epochs_path = get_epochs_path(example_subject, PIPELINE, 'feedback')
epochs = load_epochs(example_subject, PIPELINE, 'feedback', preload=False)

print(epochs_path)
print('n_epochs:', len(epochs))
print('metadata columns:', list(epochs.metadata.columns))
display(
    epochs.metadata[['context', 'cue_value', 'outcome_label']]
    .value_counts()
    .rename('n')
    .reset_index()
    .head(10)
)

In [ ]:
 run the group decoding analysis
summary_df, group_stats, auc_store = run_group_decoding(
    SUBJECTS,
    PIPELINE,
    CONTEXTS,
    window_start=WINDOW_START,
    window_end=WINDOW_END,
)

summary_df

In [ ]:
# plotting the results
pivot = (summary_df.pivot(index='subject_id', columns='context', values='mean_auc').dropna())

fig, ax = plt.subplots(figsize=(6, 5))
xpos = np.arrange(len(CONTEXTS))

for _, row in pivot.iterrows():
    ax.plot(xpos, row[CONTEXTS].values, marker="o", alpha=0.6)

means = pivot[CONTEXTS].mean(axis=0).values
sems = pivot[CONTEXTS].sem(axis=0).values

ax.errorbar(
    xpos,
    means,
    yerr=sems,
    fmt="o",
    capsize=4,
    linewidth=2,
    markersize=8,
)

ax.axhline(0.5, color="black", linestyle="--", linewidth=1)
ax.set_xticks(xpos)
ax.set_xticklabels(CONTEXTS)
ax.set_ylabel("Mean AUC")
ax.set_title(f"Outcome decoding in {WINDOW_START:.2f}-{WINDOW_END:.2f} s window")

plt.tight_layout()
plt.show()

In [ ]:
# saving the outputs
summary_path, stats_path, npz_path = save_outputs_window(
    output_dir,
    PIPELINE,
    CONTEXTS,
    summary_df,
    group_stats,
    auc_store,
)

print(summary_path)
print(stats_path)
print(npz_path)

group_stats